In [1]:
# --- Notebook Test Engine: injected setup (auto-generated) ---
import os as _os
for _k in ('AWS_ACCESS_KEY_ID','AWS_SECRET_ACCESS_KEY','AWS_SESSION_TOKEN','AWS_CREDENTIAL_EXPIRATION'):
    _os.environ.pop(_k, None)
_os.environ['AWS_DEFAULT_REGION'] = 'us-west-2'
_os.environ['AWS_REGION'] = 'us-west-2'
_nte_role = 'arn:aws:iam::674622101542:role/NotebookTestEngine-ProcessingJobRole'
try:
    from sagemaker.core.helper import session_helper as _sh
    _sh.get_execution_role = lambda *a, **k: _nte_role
except Exception:
    pass
try:
    _nte_cf = _os.environ.get('AWS_SHARED_CREDENTIALS_FILE')
    if _nte_cf and _os.path.exists(_nte_cf):
        import configparser as _cfp, datetime as _dt
        import boto3 as _b3, botocore.session as _bcs
        from botocore.credentials import RefreshableCredentials as _RC
        _nte_prof = _os.environ.get('AWS_PROFILE', 'default')
        def _nte_load():
            _cp = _cfp.ConfigParser(); _cp.read(_nte_cf)
            _s = _cp[_nte_prof]
            _tok = _s.get('aws_session_token')
            if not _tok:
                raise RuntimeError('nte: no session token in creds file')
            return {'access_key': _s['aws_access_key_id'],
                    'secret_key': _s['aws_secret_access_key'],
                    'token': _tok,
                    'expiry_time': (_dt.datetime.now(_dt.timezone.utc)
                                    + _dt.timedelta(minutes=9)).isoformat()}
        _nte_creds = _RC.create_from_metadata(metadata=_nte_load(),
                                              refresh_using=_nte_load,
                                              method='nte-shared-file')
        def _nte_get_credentials(self, *a, **k):
            # Honor sessions that were handed explicit credentials
            # (e.g. notebooks that assume_role into other accounts):
            # clobbering them would silently run every cross-account
            # client as the ambient identity. Bare/SDK-created sessions
            # (no explicit creds) still get the refreshable shared-file
            # creds so long-running waits survive credential rotation.
            _ex = getattr(self, '_credentials', None)
            if _ex is not None:
                return _ex
            return _nte_creds
        _bcs.Session.get_credentials = _nte_get_credentials
        _b3.setup_default_session(botocore_session=_bcs.get_session())
        print('nte: refreshable shared-file creds installed')
except Exception as _e:
    print('nte: refreshable-creds shim skipped:', _e)
try:
    from sagemaker.core.resources import ModelPackageGroup as _NteMPG
    _nte_mpg_create = _NteMPG.create
    def _nte_mpg_get_or_create(*_a, **_k):
        try:
            return _nte_mpg_create(*_a, **_k)
        except Exception as _e2:
            if 'already exists' in str(_e2).lower():
                _name = _k.get('model_package_group_name') or (_a[0] if _a else None)
                return _NteMPG.get(_name)
            raise
    _NteMPG.create = staticmethod(_nte_mpg_get_or_create)
    print('nte: ModelPackageGroup.create is now get-or-create')
except Exception as _e:
    print('nte: ModelPackageGroup idempotency shim skipped:', _e)


nte: ModelPackageGroup.create is now get-or-create


# EMR Serverless Step in SageMaker Pipelines

This notebook demonstrates how to use the `EMRServerlessStep` to run Spark jobs on EMR Serverless within a SageMaker Pipeline.

## Prerequisites
- An AWS account with EMR Serverless access

## What This Notebook Does
1. Creates an IAM role for EMR Serverless (if it doesn't exist)
2. Creates a sample PySpark script
3. Copies sample data to your S3 bucket
4. Creates an EMR Serverless step that provisions a new application
5. Creates and executes a SageMaker Pipeline

In [2]:
from sagemaker.mlops.workflow.emr_serverless_step import (
    EMRServerlessStep,
    EMRServerlessJobConfig,
)
from sagemaker.mlops.workflow.pipeline import Pipeline
from sagemaker.core.workflow.parameters import ParameterString
from sagemaker.core.workflow.pipeline_context import PipelineSession
from sagemaker.core.helper.session_helper import Session, get_execution_role

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Fetched defaults config from location: /opt/ml/processing/input/sm_config.yaml


In [3]:
# Create the SageMaker Session
sagemaker_session = Session()
pipeline_session = PipelineSession()
region = sagemaker_session.boto_region_name
account_id = sagemaker_session.account_id()

print(f"Region: {region}")
print(f"Account ID: {account_id}")

Region: us-west-2
Account ID: 674622101542


In [4]:
# Define variables and parameters needed for the Pipeline steps
role = get_execution_role()
default_bucket = sagemaker_session.default_bucket()
s3_prefix = "v3-emr-serverless-pipeline"

# Pipeline parameters
emr_execution_role = ParameterString(
    name="NotebookTestEngine-EMRServerlessExecutionRole",
    default_value=f"arn:aws:iam::{account_id}:role/NotebookTestEngine-EMRServerlessExecutionRole"
)

spark_script_uri = ParameterString(
    name="SparkScriptUri",
    default_value=f"s3://{default_bucket}/{s3_prefix}/scripts/spark_job.py"
)

print(f"Role: {role}")
print(f"Default Bucket: {default_bucket}")

Role: arn:aws:iam::674622101542:role/NotebookTestEngine-ProcessingJobRole
Default Bucket: sagemaker-us-west-2-674622101542


## Create IAM Role for EMR Serverless

The EMR Serverless job needs an execution role with permissions to access S3 and CloudWatch Logs.

In [5]:
import boto3
import json

# Create IAM role for EMR Serverless (if it doesn't exist)
iam_client = boto3.client('iam')

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "Service": "emr-serverless.amazonaws.com"
            },
            "Action": "sts:AssumeRole"
        }
    ]
}

try:
    iam_client.create_role(
        RoleName="NotebookTestEngine-EMRServerlessExecutionRole",
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="Execution role for EMR Serverless"
    )
    print("Role created!")
except iam_client.exceptions.EntityAlreadyExistsException:
    print("Role already exists")

# Attach required policies
for policy_arn in [
    "arn:aws:iam::aws:policy/AmazonS3FullAccess",
    "arn:aws:iam::aws:policy/CloudWatchLogsFullAccess"
]:
    iam_client.attach_role_policy(
        RoleName="NotebookTestEngine-EMRServerlessExecutionRole",
        PolicyArn=policy_arn
    )

print("NotebookTestEngine-EMRServerlessExecutionRole is ready!")
print(f"Role ARN: arn:aws:iam::{account_id}:role/NotebookTestEngine-EMRServerlessExecutionRole")

Role already exists


NotebookTestEngine-EMRServerlessExecutionRole is ready!
Role ARN: arn:aws:iam::674622101542:role/NotebookTestEngine-EMRServerlessExecutionRole


In [6]:
!mkdir -p code

In [7]:
%%writefile code/spark_job.py

"""Sample PySpark job for EMR Serverless."""
import argparse
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--input", type=str, required=True, help="Input S3 path")
    parser.add_argument("--output", type=str, required=True, help="Output S3 path")
    args = parser.parse_args()

    # Create Spark session
    spark = SparkSession.builder.appName("EMRServerlessExample").getOrCreate()

    print(f"Reading data from: {args.input}")
    
    # Define schema for abalone dataset (no headers in CSV)
    schema = StructType([
        StructField("sex", StringType(), True),
        StructField("length", DoubleType(), True),
        StructField("diameter", DoubleType(), True),
        StructField("height", DoubleType(), True),
        StructField("whole_weight", DoubleType(), True),
        StructField("shucked_weight", DoubleType(), True),
        StructField("viscera_weight", DoubleType(), True),
        StructField("shell_weight", DoubleType(), True),
        StructField("rings", DoubleType(), True),
    ])
    
    # Read input data (no header in abalone dataset)
    df = spark.read.csv(args.input, header=False, schema=schema)
    
    # Simple transformation - show schema and count
    print("Schema:")
    df.printSchema()
    print(f"Row count: {df.count()}")
    
    # Show sample data
    print("Sample data:")
    df.show(5)
    
    # Example transformation - compute statistics
    result_df = df.describe()
    
    # Write output
    print(f"Writing results to: {args.output}")
    result_df.write.mode("overwrite").parquet(args.output)
    
    print("Job completed successfully!")
    spark.stop()


if __name__ == "__main__":
    main()

Writing code/spark_job.py


In [8]:
import boto3

# Upload the Spark script to S3
s3_client = boto3.client("s3", region_name='us-west-2')
script_s3_key = f"{s3_prefix}/scripts/spark_job.py"

s3_client.upload_file(
    "code/spark_job.py",
    default_bucket,
    script_s3_key
)

script_s3_uri = f"s3://{default_bucket}/{script_s3_key}"
print(f"Spark script uploaded to: {script_s3_uri}")

Spark script uploaded to: s3://sagemaker-us-west-2-674622101542/v3-emr-serverless-pipeline/scripts/spark_job.py


## Copy Sample Data to Your Bucket

We copy the sample data from AWS public bucket to your bucket to ensure it's in the same region as EMR Serverless.

In [9]:
import boto3

# Copy sample data to your bucket (same region as EMR Serverless)
s3_resource = boto3.resource('s3', region_name='us-west-2')
copy_source = {
    'Bucket': 'notebook-test-engine-ds-674622101542-usw2',
    'Key': 'datasets/tabular/uci_abalone/abalone.csv'
}

dest_key = f"{s3_prefix}/input/abalone.csv"
s3_resource.meta.client.copy(copy_source, default_bucket, dest_key)

input_data_uri = f"s3://{default_bucket}/{dest_key}"
print(f"Sample data copied to: {input_data_uri}")

Sample data copied to: s3://sagemaker-us-west-2-674622101542/v3-emr-serverless-pipeline/input/abalone.csv


## Create EMR Serverless Step

The `EMRServerlessStep` supports two modes:
1. **Existing Application**: Use an existing EMR Serverless application ID
2. **New Application**: Create a new EMR Serverless application as part of the step

This notebook uses Option 2 (New Application) so it works out of the box.

In [10]:
# Define the EMR Serverless job configuration
job_config = EMRServerlessJobConfig(
    job_driver={
        "sparkSubmit": {
            "entryPoint": script_s3_uri,
            "entryPointArguments": [
                "--input", input_data_uri,
                "--output", f"s3://{default_bucket}/{s3_prefix}/output/"
            ],
        }
    },
    execution_role_arn=f"arn:aws:iam::{account_id}:role/NotebookTestEngine-EMRServerlessExecutionRole",
    configuration_overrides={
        "monitoringConfiguration": {
            "s3MonitoringConfiguration": {
                "logUri": f"s3://{default_bucket}/{s3_prefix}/logs/"
            }
        }
    }
)

print("EMR Serverless Job Configuration created")

EMR Serverless Job Configuration created


### Option 1: Use Existing EMR Serverless Application (Optional)

If you have an existing EMR Serverless application, you can use it instead. Uncomment the code below and replace `your-application-id` with your actual application ID.

In [11]:
# Option 1: Use an existing EMR Serverless application
# Uncomment below if you have an existing application

# step_emr_serverless_existing = EMRServerlessStep(
#     name="EMRServerlessSparkJob",
#     display_name="EMR Serverless Spark Job",
#     description="Run a PySpark job on EMR Serverless",
#     job_config=job_config,
#     application_id="your-application-id",  # Replace with your application ID
# )

print("Option 1 skipped - Using Option 2 (new application) below")

Option 1 skipped - Using Option 2 (new application) below


### Option 2: Create New EMR Serverless Application (Default)

This option creates a new EMR Serverless application as part of the pipeline step. The application will auto-start when needed and auto-stop after 15 minutes of idle time.

In [12]:
# Option 2: Create a new EMR Serverless application as part of the step
# This is the default option that works out of the box

step_emr_serverless = EMRServerlessStep(
    name="EMRServerlessSparkJob",
    display_name="EMR Serverless Spark Job",
    description="Run a PySpark job with a newly created EMR Serverless application",
    job_config=job_config,
    application_config={
        "name": "sagemaker-pipeline-spark-app",
        "releaseLabel": "emr-6.15.0",
        "type": "SPARK",
        "autoStartConfiguration": {
            "enabled": True
        },
        "autoStopConfiguration": {
            "enabled": True,
            "idleTimeoutMinutes": 15
        },
    },
)

print(f"Step Name: {step_emr_serverless.name}")
print(f"Step Type: {step_emr_serverless.step_type}")

Step Name: EMRServerlessSparkJob
Step Type: StepTypeEnum.EMR_SERVERLESS


In [13]:
# Create the pipeline

pipeline = Pipeline(
    name="EMRServerlessPipeline",
    parameters=[
        emr_execution_role,
        spark_script_uri,
    ],
    steps=[step_emr_serverless],
    sagemaker_session=pipeline_session,
)

print("Pipeline created successfully!")

Pipeline created successfully!


In [14]:
import json

definition = json.loads(pipeline.definition())
print(json.dumps(definition, indent=2))

{
  "Version": "2020-12-01",
  "Metadata": {},
  "Parameters": [
    {
      "Name": "NotebookTestEngine-EMRServerlessExecutionRole",
      "Type": "String",
      "DefaultValue": "arn:aws:iam::674622101542:role/NotebookTestEngine-EMRServerlessExecutionRole"
    },
    {
      "Name": "SparkScriptUri",
      "Type": "String",
      "DefaultValue": "s3://sagemaker-us-west-2-674622101542/v3-emr-serverless-pipeline/scripts/spark_job.py"
    }
  ],
  "PipelineExperimentConfig": {
    "ExperimentName": {
      "Get": "Execution.PipelineName"
    },
    "TrialName": {
      "Get": "Execution.PipelineExecutionId"
    }
  },
  "MlflowConfig": null,
  "Steps": [
    {
      "Name": "EMRServerlessSparkJob",
      "Type": "EMRServerless",
      "Arguments": {
        "ExecutionRoleArn": "arn:aws:iam::674622101542:role/NotebookTestEngine-EMRServerlessExecutionRole",
        "JobConfig": {
          "executionRoleArn": "arn:aws:iam::674622101542:role/NotebookTestEngine-EMRServerlessExecutionRole",


## Execute Pipeline

The cells below will:
1. Create/update the pipeline in SageMaker
2. Start the pipeline execution
3. Wait for completion

In [15]:
# Create/update the pipeline
pipeline.upsert(role_arn=role)
print("Pipeline upserted successfully!")

[07/21/26 07:13:57] INFO     Role                                                          ]8;id=9837898;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=9837899;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#598\598]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' validated for pipeline. Using it.                                            

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=9837906;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=9837907;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#288\288]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

[07/21/26 07:13:58] INFO     Role                                                          ]8;id=9837912;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=9837913;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#598\598]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' validated for pipeline. Using it.                                            

Pipeline upserted successfully!


In [16]:
# Start the pipeline execution
execution = pipeline.start()
print(f"Pipeline execution started: {execution.arn}")

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=9837918;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=9837919;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#288\288]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

Pipeline execution started: arn:aws:sagemaker:us-west-2:674622101542:pipeline/EMRServerlessPipeline/execution/w8lwgg2p15ly


In [17]:
import time

# Wait for pipeline execution to complete
while True:
    status = execution.describe()['PipelineExecutionStatus']
    print(f"Status: {status}")
    
    if status in ['Succeeded', 'Failed', 'Stopped']:
        print(f"Pipeline finished with status: {status}")
        break
    
    print("Still running... waiting 30 seconds")
    time.sleep(30)

Status: Executing
Still running... waiting 30 seconds


Status: Executing
Still running... waiting 30 seconds


Status: Executing
Still running... waiting 30 seconds


Status: Executing
Still running... waiting 30 seconds


Status: Executing
Still running... waiting 30 seconds


Status: Executing
Still running... waiting 30 seconds


Status: Succeeded
Pipeline finished with status: Succeeded


In [18]:
# Check step execution details
steps = execution.list_steps()
for step in steps:
    print(f"Step: {step['StepName']}")
    print(f"  Status: {step['StepStatus']}")
    if 'FailureReason' in step:
        print(f"  Failure Reason: {step['FailureReason']}")
    print()

Step: EMRServerlessSparkJob
  Status: Succeeded



## Cleanup (Optional)

Uncomment the cell below to delete the pipeline when you're done.

In [19]:
# Uncomment to delete the pipeline
sm_client = sagemaker_session.sagemaker_client
sm_client.delete_pipeline(PipelineName="EMRServerlessPipeline")
print("Pipeline deleted")

Pipeline deleted
